# Module 39 — Cohort Analysis: Retention Heatmaps

> **Track 4 · Marketing Analytics & Optimization**  
> **Difficulty**: Intermediate  
> **Runtime**: ~5 minutes  
> **Key Libraries**: Pandas, Seaborn, Plotly  
> **Prerequisite**: Module 38 (RFM Segmentation)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic User Data](#3-synthetic-user-data)
4. [Cohort Assignment](#4-cohort-assignment)
5. [Retention Calculation](#5-retention-calculation)
6. [Visualization](#6-visualization)
7. [Key Business Takeaway](#7-key-business-takeaway)

---
## §1 · Business Context

### Why cohort analysis?

Cohort analysis tracks **user behavior over time**:
- **Retention**: How many users return after signup.
- **Churn**: When users stop engaging.
- **Seasonality**: How different cohorts behave.

**Applications**:
1. **Product launches**: Measure retention by signup month.
2. **Marketing campaigns**: Track cohort performance.
3. **Feature releases**: See impact on user retention.

**Key metrics**:
- **Retention rate**: % of users active after N periods.
- **Churn rate**: % of users who stopped engaging.
- **Lifetime value**: Revenue per user over time.

In this notebook we will:
1. Generate synthetic user activity data.
2. Assign users to monthly cohorts.
3. Calculate retention rates by cohort.
4. Visualize retention heatmaps.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic User Data

In [ ]:
# Generate synthetic user activity data
N_USERS = 5000
N_MONTHS = 12

# Generate user signup dates
users = pd.DataFrame({
    'user_id': range(N_USERS),
    'signup_date': pd.date_range('2023-01-01', periods=N_USERS, freq='D')
})

# Generate activity logs
activities = []
for _, user in users.iterrows():
    # Generate random activity months
    n_active_months = np.random.randint(1, N_MONTHS + 1)
    active_months = np.random.choice(range(N_MONTHS), size=n_active_months, replace=False)
    
    for month in active_months:
        activity_date = user['signup_date'] + pd.DateOffset(months=month)
        activities.append({
            'user_id': user['user_id'],
            'activity_date': activity_date,
            'signup_date': user['signup_date']
        })

df_activities = pd.DataFrame(activities)

print(f"✓ Generated activity data")
print(f"  Users: {N_USERS:,}")
print(f"  Activities: {len(df_activities):,}")
print(f"  Date range: {df_activities['activity_date'].min()} to {df_activities['activity_date'].max()}")

---
## §4 · Cohort Assignment

In [ ]:
# Assign cohorts based on signup month
df_activities['cohort'] = df_activities['signup_date'].dt.to_period('M')
df_activities['activity_month'] = df_activities['activity_date'].dt.to_period('M')

# Calculate months since signup
df_activities['months_since_signup'] = (
    (df_activities['activity_date'].dt.year - df_activities['signup_date'].dt.year) * 12 +
    (df_activities['activity_date'].dt.month - df_activities['signup_date'].dt.month)
)

print(f"✓ Cohort assignment complete")
print(f"  Cohorts: {df_activities['cohort'].nunique()}")
print(f"  Max months since signup: {df_activities['months_since_signup'].max()}")

---
## §5 · Retention Calculation

In [ ]:
# Calculate retention by cohort
cohort_data = df_activities.groupby(['cohort', 'months_since_signup'])['user_id'].nunique().reset_index()
cohort_data.columns = ['cohort', 'months_since_signup', 'active_users']

# Get cohort sizes
cohort_sizes = df_activities.groupby('cohort')['user_id'].nunique().reset_index()
cohort_sizes.columns = ['cohort', 'cohort_size']

# Merge
cohort_data = cohort_data.merge(cohort_sizes, on='cohort')

# Calculate retention rate
cohort_data['retention_rate'] = cohort_data['active_users'] / cohort_data['cohort_size'] * 100

# Create retention matrix
retention_matrix = cohort_data.pivot(
    index='cohort',
    columns='months_since_signup',
    values='retention_rate'
)

print(f"✓ Retention matrix calculated")
print(f"  Shape: {retention_matrix.shape}")
print(f"\nSample retention rates:")
print(retention_matrix.head())

---
## §6 · Visualization

In [ ]:
# Create retention heatmap
fig = px.imshow(
    retention_matrix.values,
    labels=dict(x="Months Since Signup", y="Cohort", color="Retention Rate (%)") ,
    x=[f"Month {i}" for i in retention_matrix.columns],
    y=[str(c) for c in retention_matrix.index],
    title='Cohort Retention Heatmap',
    color_continuous_scale='Blues',
    aspect='auto'
)

fig.update_layout(height=500)
fig.show()

# Average retention curve
avg_retention = retention_matrix.mean()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=list(range(len(avg_retention))),
    y=avg_retention.values,
    mode='lines+markers',
    name='Average Retention'
))

fig2.update_layout(
    title='Average Retention Curve',
    xaxis_title='Months Since Signup',
    yaxis_title='Retention Rate (%)',
    height=400
)
fig2.show()

print("💡 Key insights:")
print("   - Retention drops significantly after month 1")
print("   - Later cohorts may show different patterns")
print("   - Identify which cohorts retain best")

---
## §7 · Key Business Takeaway

> ### 🎯 Action Items for Product & Marketing Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Product launches, campaign analysis, retention monitoring |
> | **Cohorts** | Monthly cohorts for most businesses |
> | **Metrics** | Retention rate, churn rate, LTV by cohort |
> | **Visualization** | Heatmaps for overview, curves for trends |
> | **Action** | Focus on improving month 1 retention |
>
> ### 💡 Retention Improvement Strategies
>
> | Timeframe | Strategy | Expected Impact |
> |-----------|----------|------------------|
> | Month 1 | Onboarding optimization | +10-20% retention |
> | Month 2-3 | Engagement features | +5-10% retention |
> | Month 6+ | Loyalty programs | +3-5% retention |
>
> ### 🔑 Key Takeaways
>
> 1. **Cohort analysis reveals retention patterns** over time.
> 2. **Month 1 retention** is the most critical metric.
> 3. **Heatmaps provide clear visualization** of retention trends.
> 4. **Different cohorts may behave differently** due to seasonality.
> 5. **Focus retention efforts** on early lifecycle stages.